# Summary_Day11_online.ipynb  
## CNN 기반 이미지 분류 · 인터넷 가능 버전 · MNIST + CIFAR-10

이번 11강은 **CNN 기반 이미지 분류**를 시작하는 강의다.

10강까지는 이미지를 1차원 벡터로 펼쳐서 MLP에 넣었다.  
하지만 강사님은 CNN으로 넘어오면서 가장 먼저 이 문제를 강조했다.

```text
이미지를 1차원으로 펼치면 공간 정보가 사라진다.
```

이미지에서는 픽셀의 위치 관계가 중요하다.  
눈 옆에 코가 있고, 선과 모서리와 질감이 주변 픽셀과 함께 의미를 만든다.  
그런데 완전 결합형 DNN은 이미지를 처음부터 `[3072]` 같은 1차원 벡터로 펼쳐 버리기 때문에, 이웃 픽셀 간의 구조를 잃는다.

CNN은 이 문제를 해결하기 위해 이미지를 2차원/3차원 구조 그대로 보면서 작은 필터를 움직여 특징을 뽑는다.

강의 핵심 흐름은 다음이다.

```text
이미지의 공간 구조 문제
→ CNN = Convolutional Neural Network
→ 특징 추출 Feature Extraction
→ 분류 Classifier
→ Kernel / Filter
→ Feature Map
→ Conv2d
→ ReLU
→ MaxPool2d
→ Flatten
→ Fully Connected Layer
→ CIFAR-10에서 FCN과 CNN 비교
```

이 파일은 **인터넷이 되는 환경**을 기준으로 한다.  
MNIST와 CIFAR-10을 `torchvision.datasets`로 다운로드해서 강의 흐름을 그대로 따라간다.

> 필기 포인트:  
> CNN은 이미지를 직접 학습한다기보다, 이미지에서 어떤 특징을 볼지 결정하는 **필터/커널을 학습한다**고 이해하면 된다.

## 1. 전체 실습 목적

이번 실습의 목적은 다음이다.

1. MLP 방식이 왜 이미지의 공간 정보를 잃는지 이해한다.
2. `nn.Conv2d`가 이미지 위를 훑으며 feature map을 만드는 흐름을 본다.
3. `kernel`, `filter`, `feature map`, `channel`의 의미를 정리한다.
4. `Conv2d → ReLU → Conv2d → ReLU → MaxPool2d`의 shape 변화를 확인한다.
5. CNN의 앞부분은 특징 추출기, 뒷부분은 분류기라는 구조를 이해한다.
6. CIFAR-10 이미지를 완전 결합형 모델과 CNN 모델로 각각 학습한다.
7. 왜 CNN이 이미지 분류에 더 적합한지 코드와 그래프로 비교한다.

## 2. 라이브러리 준비

### 함수/모듈 사용법

```python
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.datasets as datasets
import torchvision.transforms as transforms
```

- `torch`: Tensor와 자동 미분을 사용한다.
- `nn`: `Conv2d`, `MaxPool2d`, `Linear`, `ReLU` 같은 Layer를 만든다.
- `optim`: Optimizer를 만든다.
- `datasets`: MNIST, CIFAR-10 같은 데이터셋을 불러온다.
- `transforms`: 이미지 전처리를 순서대로 구성한다.
- `DataLoader`: 데이터를 mini-batch 단위로 꺼낸다.

> 실습 메모:  
> 원본 노트북에는 한글 폰트 설치, `torchviz`, `torchinfo` 설치 코드가 있다.  
> 이 Summary는 실행 안정성을 위해 외부 설치가 꼭 필요한 시각화 도구는 제외하고 핵심 CNN 흐름을 정리한다.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

import torchvision.datasets as datasets
import torchvision.transforms as transforms

from torch.utils.data import DataLoader, Subset
from sklearn.metrics import classification_report, confusion_matrix

%matplotlib inline

torch.manual_seed(123)
np.random.seed(123)

plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["axes.unicode_minus"] = False

print("PyTorch:", torch.__version__)

## 3. device 설정

CNN은 이미지 연산이 많기 때문에 GPU가 있으면 GPU를 쓰는 것이 좋다.

### 함수 사용법

```python
torch.cuda.is_available()
torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model.to(device)
tensor.to(device)
```

- `torch.cuda.is_available()`은 CUDA GPU 사용 가능 여부를 확인한다.
- `.to(device)`는 모델이나 Tensor를 CPU/GPU로 이동한다.
- 모델과 Tensor가 서로 다른 device에 있으면 연산 에러가 난다.

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("device:", device)

## 4. ReLU 함수 복습

CNN에서도 ReLU는 계속 사용된다.

ReLU는 음수는 0으로 만들고 양수는 그대로 통과시킨다.

### 함수 사용법

```python
relu = nn.ReLU()
y = relu(x)
```

- `x`: 입력 Tensor다.
- `y`: ReLU를 지난 출력 Tensor다.
- Conv2d 뒤에 ReLU를 넣어 비선형성을 추가한다.

In [ ]:
relu = nn.ReLU()

x_np = np.arange(-2.0, 2.1, 0.25)
x = torch.tensor(x_np).float()
y = relu(x)

plt.plot(x.numpy(), y.numpy())
plt.xlabel("x")
plt.ylabel("ReLU(x)")
plt.title("ReLU Function")
plt.show()

## 5. CNN이 필요한 이유

완전 결합형 DNN은 이미지를 처음부터 1차원 벡터로 펼친다.

예를 들어 CIFAR-10 이미지는 다음 구조다.

```text
3 × 32 × 32 = 3072
```

MLP는 이 이미지를 `[3072]` 벡터로 만든다.

문제는 이 과정에서 다음 정보가 약해진다는 것이다.

- 픽셀끼리 가까이 있다는 정보
- 선과 모서리의 방향
- 질감과 패턴
- 지역적인 형태

CNN은 이미지를 그대로 두고 작은 필터를 움직여 지역적 특징을 뽑는다.

> 강의식 표현:  
> CNN은 돋보기처럼 이미지의 특정 부분을 훑으면서 특징을 찾는 방식이다.

## 6. CNN의 2단계 구조

강사님이 강조한 CNN의 큰 구조는 다음 두 단계다.

```text
1. Feature Extraction, 특징 추출
2. Classifier, 분류
```

앞부분은 이미지에서 선, 모서리, 질감 같은 특징을 뽑는다.

```text
Conv2d → ReLU → Conv2d → ReLU → MaxPool2d
```

뒷부분은 뽑힌 특징을 바탕으로 class를 판단한다.

```text
Flatten → Linear → ReLU → Linear
```

> 시험 포인트:  
> CNN은 특징 추출기와 분류기로 나누어 이해하면 구조가 훨씬 쉬워진다.

In [ ]:
cnn_parts = {
    "Feature Extraction": "Conv2d, ReLU, Pooling으로 이미지 특징을 뽑는다",
    "Classifier": "Flatten과 Linear로 최종 class를 예측한다"
}

for key, value in cnn_parts.items():
    print(f"{key}: {value}")

## 7. MNIST 이미지로 Conv2d 감각 잡기

강의 첫 번째 CNN 실습은 MNIST 이미지 한 장에 직접 Conv2d를 적용하는 흐름이다.

MNIST 한 장은 흑백 이미지라 channel이 1개다.

```text
원래 shape: [1, 28, 28]
Conv2d 입력 shape: [N, C, H, W]
한 장만 넣을 때: [1, 1, 28, 28]
```

### 함수 사용법

```python
datasets.MNIST(root="./data", train=True, download=True, transform=transform)
```

- `download=True`: 인터넷에서 데이터를 받는다.
- `transform=transforms.ToTensor()`: 이미지를 Tensor로 바꾼다.

In [ ]:
data_root = "./data"

mnist_transform = transforms.Compose([
    transforms.ToTensor()
])

mnist_train = datasets.MNIST(
    root=data_root,
    train=True,
    download=True,
    transform=mnist_transform
)

image, label = mnist_train[0]

print("label:", label)
print("image shape:", image.shape)

In [ ]:
plt.imshow(image.squeeze(0), cmap="gray_r")
plt.title(f"MNIST label = {label}")
plt.axis("off")
plt.show()

## 8. Conv2d 입력을 NCHW로 맞추기

PyTorch CNN의 입력은 4차원이다.

```text
[N, C, H, W]
```

- `N`: batch size, 이미지 장수다.
- `C`: channel 수다.
- `H`: height다.
- `W`: width다.

MNIST 한 장은 `[1, 28, 28]`이므로 batch 차원 하나를 추가해 `[1, 1, 28, 28]`로 만든다.

### 함수 사용법

```python
image.view(1, 1, 28, 28)
```

- Tensor shape을 바꾼다.
- 값의 순서는 그대로 두고 모양만 바꾼다.

In [ ]:
image_4d = image.view(1, 1, 28, 28)

print("image_4d shape:", image_4d.shape)

## 9. 대각선 필터 Conv2d 만들기

강의에서는 대각선 방향에 반응하는 특수한 3×3 필터를 직접 만들었다.

### 함수 사용법

```python
nn.Conv2d(in_channels, out_channels, kernel_size)
```

- `in_channels`: 입력 이미지의 channel 수다.
- `out_channels`: 만들 feature map 수다.
- `kernel_size`: 필터 크기다.

여기서는 흑백 이미지 1채널을 받아 feature map 1개를 만든다.

```python
nn.Conv2d(1, 1, 3)
```

In [ ]:
conv_diag = nn.Conv2d(1, 1, 3)

nn.init.constant_(conv_diag.bias, 0.0)

w_np = np.array([
    [0, 0, 1],
    [0, 1, 0],
    [1, 0, 0]
])

w = torch.tensor(w_np).float().view(1, 1, 3, 3)
conv_diag.weight.data = w

print("conv weight shape:", conv_diag.weight.shape)
print(conv_diag.weight.data)

## 10. Conv2d를 여러 번 적용하기

3×3 kernel을 padding 없이 적용하면 이미지 크기가 줄어든다.

```text
28×28 → 26×26 → 24×24 → 22×22
```

커널이 가장자리 밖으로 나가지 못하기 때문이다.

In [ ]:
w1 = conv_diag(image_4d)
w2 = conv_diag(w1)
w3 = conv_diag(w2)

conv_images = [image_4d, w1, w2, w3]

for i, img in enumerate(conv_images):
    print(f"{i}번째 shape:", img.shape)

In [ ]:
plt.figure(figsize=(8, 2))

for i, img in enumerate(conv_images):
    ax = plt.subplot(1, 4, i + 1)
    arr = img.detach().numpy().reshape(img.shape[-2], img.shape[-1])
    plt.imshow(arr, cmap="gray_r")
    plt.title(f"step {i}")
    plt.axis("off")

plt.tight_layout()
plt.show()

그래프 해석:

- Conv2d를 반복할수록 이미지가 작아진다.
- 필터가 강조하는 방향의 특징이 더 도드라진다.
- 원본 이미지는 그대로이고, 필터를 통과한 결과가 feature map이다.

> 핵심 문장:  
> CNN은 이미지 자체를 바꾸는 것이 아니라, 필터를 통해 특징 맵을 만들어 간다.

## 11. Conv2d / ReLU / MaxPool2d shape 흐름

이번에는 CIFAR-10 같은 RGB 이미지 구조를 가정하고 dummy Tensor로 shape 변화를 본다.

```text
입력: [100, 3, 32, 32]
Conv2d(3, 32, 3): [100, 32, 30, 30]
ReLU: shape 유지
Conv2d(32, 32, 3): [100, 32, 28, 28]
ReLU: shape 유지
MaxPool2d(2, 2): [100, 32, 14, 14]
```

### 함수 사용법

```python
nn.Conv2d(3, 32, 3)
nn.MaxPool2d((2, 2))
```

- `Conv2d(3, 32, 3)`은 RGB 3채널을 받아 32개 feature map을 만든다.
- `MaxPool2d((2,2))`는 2×2 영역에서 가장 큰 값만 남긴다.

In [ ]:
conv1 = nn.Conv2d(3, 32, 3)
relu = nn.ReLU(inplace=True)
conv2 = nn.Conv2d(32, 32, 3)
maxpool = nn.MaxPool2d((2, 2))

inputs_dummy = torch.randn(100, 3, 32, 32)

x1 = conv1(inputs_dummy)
x2 = relu(x1)
x3 = conv2(x2)
x4 = relu(x3)
x5 = maxpool(x4)

print("inputs:", inputs_dummy.shape)
print("x1 conv1:", x1.shape)
print("x2 relu:", x2.shape)
print("x3 conv2:", x3.shape)
print("x4 relu:", x4.shape)
print("x5 maxpool:", x5.shape)

## 12. Conv2d weight shape 이해

Conv2d의 weight shape은 다음 순서다.

```text
[out_channels, in_channels, kernel_height, kernel_width]
```

예를 들어 `nn.Conv2d(3, 32, 3)`이면 다음과 같다.

```text
[32, 3, 3, 3]
```

- 32개 필터가 있다.
- 각 필터는 RGB 3채널을 모두 본다.
- 각 필터 크기는 3×3이다.

In [ ]:
print("conv1 weight shape:", conv1.weight.shape)
print("conv1 bias shape:", conv1.bias.shape)

print("conv2 weight shape:", conv2.weight.shape)
print("conv2 bias shape:", conv2.bias.shape)

## 13. nn.Sequential로 특징 추출기 만들기

Conv2d와 ReLU와 MaxPool을 순서대로 쌓을 때 `nn.Sequential`을 사용하면 코드가 짧아진다.

### 함수 사용법

```python
features = nn.Sequential(
    conv1,
    relu,
    conv2,
    relu,
    maxpool
)
```

- 입력 Tensor가 Layer를 순서대로 통과한다.
- 특징 추출기처럼 한 덩어리로 관리할 수 있다.

In [ ]:
features = nn.Sequential(
    nn.Conv2d(3, 32, 3),
    nn.ReLU(inplace=True),
    nn.Conv2d(32, 32, 3),
    nn.ReLU(inplace=True),
    nn.MaxPool2d((2, 2))
)

feature_outputs = features(inputs_dummy)

print("feature_outputs shape:", feature_outputs.shape)

## 14. Flatten으로 분류기 입력 만들기

CNN 특징 추출기를 통과한 결과는 여전히 4차원 Tensor다.

```text
[100, 32, 14, 14]
```

Linear Layer에 넣기 위해 batch 차원을 제외한 나머지를 하나로 펼친다.

```text
32 × 14 × 14 = 6272
```

### 함수 사용법

```python
nn.Flatten()
```

- batch 차원은 유지한다.
- 나머지 차원을 모두 곱해서 1차원 feature로 만든다.

In [ ]:
flatten = nn.Flatten()

flat_outputs = flatten(feature_outputs)

print("Flatten 이전:", feature_outputs.shape)
print("Flatten 이후:", flat_outputs.shape)
print("한 장당 feature 수:", 32 * 14 * 14)

> 시험 포인트:  
> `MaxPool2d` 뒤에 나온 `[32, 14, 14]`를 Flatten하면 6272개 feature가 된다.  
> 그래서 classifier의 첫 Linear 입력은 `nn.Linear(6272, n_hidden)`이 된다.

## 15. 공통 함수 정의하기

강의 노트에서는 반복되는 학습 코드를 공통 함수로 분리했다.

여기서는 다음 함수를 만든다.

```text
torch_seed()
eval_loss()
fit()
evaluate_history()
show_images_labels()
```

함수로 분리하면 FCN과 CNN을 같은 방식으로 학습하고 비교할 수 있다.

In [ ]:
def torch_seed(seed=123):
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def eval_loss(loader, device, net, criterion):
    net.eval()

    with torch.no_grad():
        for images, labels in loader:
            inputs = images.to(device)
            labels = labels.to(device)
            outputs = net(inputs)
            loss = criterion(outputs, labels)
            break

    return loss


def fit(net, optimizer, criterion, num_epochs, train_loader, test_loader, device):
    history = []

    for epoch in range(num_epochs):
        net.train()

        train_loss = 0.0
        train_acc = 0.0
        n_train = 0

        for inputs, labels in train_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = net(inputs)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            predicted = torch.max(outputs, 1)[1]

            train_loss += loss.item() * labels.size(0)
            train_acc += (predicted == labels).sum().item()
            n_train += labels.size(0)

        net.eval()

        val_loss = 0.0
        val_acc = 0.0
        n_val = 0

        with torch.no_grad():
            for inputs, labels in test_loader:
                inputs = inputs.to(device)
                labels = labels.to(device)

                outputs = net(inputs)
                loss = criterion(outputs, labels)

                predicted = torch.max(outputs, 1)[1]

                val_loss += loss.item() * labels.size(0)
                val_acc += (predicted == labels).sum().item()
                n_val += labels.size(0)

        history.append([
            epoch + 1,
            train_loss / n_train,
            train_acc / n_train,
            val_loss / n_val,
            val_acc / n_val
        ])

        print(
            f"epoch {epoch + 1} | "
            f"train_loss={history[-1][1]:.4f} | train_acc={history[-1][2]:.4f} | "
            f"val_loss={history[-1][3]:.4f} | val_acc={history[-1][4]:.4f}"
        )

    return np.array(history)


def evaluate_history(history, title="Learning Curve"):
    print(f"초기 검증 손실: {history[0, 3]:.5f}, 초기 검증 정확도: {history[0, 4]:.5f}")
    print(f"최종 검증 손실: {history[-1, 3]:.5f}, 최종 검증 정확도: {history[-1, 4]:.5f}")

    plt.plot(history[:, 0], history[:, 1], label="train loss")
    plt.plot(history[:, 0], history[:, 3], label="val loss")
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.title(title + " Loss")
    plt.legend()
    plt.show()

    plt.plot(history[:, 0], history[:, 2], label="train acc")
    plt.plot(history[:, 0], history[:, 4], label="val acc")
    plt.xlabel("epoch")
    plt.ylabel("accuracy")
    plt.title(title + " Accuracy")
    plt.legend()
    plt.show()

In [ ]:
def show_images_labels(loader, classes, net=None, device=None, n_show=20):
    images, labels = next(iter(loader))

    predicted = None

    if net is not None:
        net.eval()

        with torch.no_grad():
            inputs = images.to(device)
            outputs = net(inputs)
            predicted = torch.max(outputs, 1)[1].cpu()

    plt.figure(figsize=(10, 4))

    for i in range(min(n_show, len(images))):
        ax = plt.subplot(2, 10, i + 1)

        img = images[i]

        if img.ndim == 1:
            img = img.view(3, 32, 32)

        img = img.permute(1, 2, 0).numpy()
        img = (img * 0.5) + 0.5
        img = np.clip(img, 0, 1)

        plt.imshow(img)

        label_name = classes[labels[i].item()]

        if predicted is None:
            title = label_name
            color = "black"
        else:
            pred_name = classes[predicted[i].item()]
            title = f"{label_name}\n→ {pred_name}"
            color = "black" if predicted[i].item() == labels[i].item() else "red"

        plt.title(title, fontsize=8, color=color)
        plt.axis("off")

    plt.tight_layout()
    plt.show()

## 16. CIFAR-10 데이터 전처리

CIFAR-10은 10개 class의 컬러 이미지 데이터다.

```text
이미지 크기: 3 × 32 × 32
class 수: 10개
class 이름: plane, car, bird, cat, deer, dog, frog, horse, ship, truck
```

강의에서는 두 가지 전처리를 비교한다.

```text
FCN용: ToTensor → Normalize → Flatten
CNN용: ToTensor → Normalize
```

CNN용은 공간 구조를 보존해야 하므로 Flatten하지 않는다.

In [ ]:
transform_fcn = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    transforms.Lambda(lambda x: x.view(-1))
])

transform_cnn = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

classes = (
    "plane", "car", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
)

print("FCN transform:", transform_fcn)
print("CNN transform:", transform_cnn)

## 17. CIFAR-10 다운로드

인터넷이 가능한 환경에서는 `download=True`로 CIFAR-10을 내려받는다.

### 함수 사용법

```python
datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
```

- `train=True`: 훈련 데이터다.
- `train=False`: 검증/테스트 데이터다.
- `transform`: 전처리 방법이다.

In [ ]:
train_set_fcn_full = datasets.CIFAR10(
    root=data_root,
    train=True,
    download=True,
    transform=transform_fcn
)

test_set_fcn_full = datasets.CIFAR10(
    root=data_root,
    train=False,
    download=True,
    transform=transform_fcn
)

train_set_cnn_full = datasets.CIFAR10(
    root=data_root,
    train=True,
    download=True,
    transform=transform_cnn
)

test_set_cnn_full = datasets.CIFAR10(
    root=data_root,
    train=False,
    download=True,
    transform=transform_cnn
)

print("CIFAR-10 train:", len(train_set_cnn_full))
print("CIFAR-10 test:", len(test_set_cnn_full))

## 18. 빠른 실행용 Subset 만들기

원본 강의는 전체 CIFAR-10과 50 epoch를 사용한다.

이 Summary는 실행 시간을 줄이기 위해 일부 데이터와 적은 epoch를 기본으로 둔다.  
원본처럼 더 길게 학습하려면 `train_size`, `test_size`, `num_epochs` 값을 늘리면 된다.

In [ ]:
train_size = 5000
test_size = 1000

train_indices = list(range(train_size))
test_indices = list(range(test_size))

train_set_fcn = Subset(train_set_fcn_full, train_indices)
test_set_fcn = Subset(test_set_fcn_full, test_indices)

train_set_cnn = Subset(train_set_cnn_full, train_indices)
test_set_cnn = Subset(test_set_cnn_full, test_indices)

batch_size = 100

train_loader_fcn = DataLoader(train_set_fcn, batch_size=batch_size, shuffle=True)
test_loader_fcn = DataLoader(test_set_fcn, batch_size=batch_size, shuffle=False)

train_loader_cnn = DataLoader(train_set_cnn, batch_size=batch_size, shuffle=True)
test_loader_cnn = DataLoader(test_set_cnn, batch_size=batch_size, shuffle=False)

print("FCN train batch:", len(train_loader_fcn))
print("CNN train batch:", len(train_loader_cnn))

## 19. FCN용 데이터와 CNN용 데이터 shape 비교

같은 CIFAR-10 이미지라도 전처리에 따라 입력 shape이 다르다.

```text
FCN용: [batch, 3072]
CNN용: [batch, 3, 32, 32]
```

In [ ]:
images_fcn, labels_fcn = next(iter(train_loader_fcn))
images_cnn, labels_cnn = next(iter(train_loader_cnn))

print("FCN images shape:", images_fcn.shape)
print("CNN images shape:", images_cnn.shape)
print("labels shape:", labels_cnn.shape)

In [ ]:
show_images_labels(test_loader_cnn, classes, net=None, device=None, n_show=20)

## 20. 완전 결합형 FCN 모델 정의

FCN은 이미지를 `[3072]` 벡터로 펼쳐서 분류한다.

구조는 다음이다.

```text
Linear(3072 → 128)
→ ReLU
→ Linear(128 → 10)
```

공간 정보를 이미 잃은 상태에서 학습한다는 한계가 있다.

In [ ]:
class FCNNet(nn.Module):
    def __init__(self, n_input, n_hidden, n_output):
        super().__init__()

        self.l1 = nn.Linear(n_input, n_hidden)
        self.relu = nn.ReLU(inplace=True)
        self.l2 = nn.Linear(n_hidden, n_output)

    def forward(self, x):
        x1 = self.l1(x)
        x2 = self.relu(x1)
        x3 = self.l2(x2)
        return x3

n_input_fcn = 3 * 32 * 32
n_hidden = 128
n_output = 10

torch_seed()

fcn_net = FCNNet(n_input_fcn, n_hidden, n_output).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(fcn_net.parameters(), lr=0.01)

print(fcn_net)

## 21. FCN 모델 학습

원본 강의에서는 50 epoch를 돌려 FCN과 CNN을 비교한다.  
여기서는 실행 시간을 줄여 2 epoch만 기본으로 둔다.

In [ ]:
num_epochs = 2

history_fcn = fit(
    fcn_net,
    optimizer,
    criterion,
    num_epochs,
    train_loader_fcn,
    test_loader_fcn,
    device
)

evaluate_history(history_fcn, title="FCN")

## 22. CNN 모델 정의

CNN은 특징 추출기와 분류기를 나누어 정의한다.

```text
features:
Conv2d(3 → 32, 3)
→ ReLU
→ Conv2d(32 → 32, 3)
→ ReLU
→ MaxPool2d(2,2)

classifier:
Flatten
→ Linear(32*14*14 → 128)
→ ReLU
→ Linear(128 → 10)
```

### 함수 사용법

```python
self.features = nn.Sequential(...)
self.classifier = nn.Sequential(...)
```

- `features`는 이미지의 특징을 뽑는다.
- `classifier`는 뽑힌 특징으로 class를 판단한다.

In [ ]:
class CNNNet(nn.Module):
    def __init__(self, n_output, n_hidden):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3),
            nn.ReLU(inplace=True),
            nn.MaxPool2d((2, 2))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 14 * 14, n_hidden),
            nn.ReLU(inplace=True),
            nn.Linear(n_hidden, n_output)
        )

    def forward(self, x):
        x1 = self.features(x)
        x2 = self.classifier(x1)
        return x2

torch_seed()

cnn_net = CNNNet(n_output=10, n_hidden=128).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(cnn_net.parameters(), lr=0.01)

print(cnn_net)

## 23. CNN 모델 shape 확인

CNN에 dummy input을 넣어 각 단계의 shape을 확인한다.

In [ ]:
dummy = torch.randn(100, 3, 32, 32).to(device)

with torch.no_grad():
    feature_out = cnn_net.features(dummy)
    logits_out = cnn_net(dummy)

print("dummy input:", dummy.shape)
print("feature_out:", feature_out.shape)
print("logits_out:", logits_out.shape)

해석:

```text
feature_out = [100, 32, 14, 14]
logits_out = [100, 10]
```

- 특징 추출기 결과는 아직 4차원이다.
- 분류기 결과는 class 10개 logits다.

## 24. CNN 모델 학습

CNN은 이미지의 공간 구조를 유지한 상태로 특징을 추출한다.  
그래서 FCN보다 이미지 분류에 더 적합하다.

In [ ]:
num_epochs = 2

history_cnn = fit(
    cnn_net,
    optimizer,
    criterion,
    num_epochs,
    train_loader_cnn,
    test_loader_cnn,
    device
)

evaluate_history(history_cnn, title="CNN")

## 25. FCN과 CNN 성능 비교

강의에서는 CIFAR-10에서 완전 결합형 모델보다 CNN이 더 좋은 결과를 내는 흐름을 보여준다.

핵심 이유는 다음이다.

```text
FCN: 이미지를 1차원으로 펼쳐 공간 정보 손실
CNN: 2차원 공간 구조를 유지하면서 필터로 특징 추출
```

In [ ]:
print("FCN 최종 검증 정확도:", history_fcn[-1, 4])
print("CNN 최종 검증 정확도:", history_cnn[-1, 4])

plt.bar(["FCN", "CNN"], [history_fcn[-1, 4], history_cnn[-1, 4]])
plt.ylabel("validation accuracy")
plt.title("FCN vs CNN")
plt.show()

그래프 해석:

- CNN이 더 높게 나오면 이미지의 공간 구조를 유지한 효과를 볼 수 있다.
- 짧은 epoch에서는 결과가 흔들릴 수 있다.
- 원본 강의처럼 더 오래 학습하면 CNN의 장점이 더 잘 드러난다.

## 26. CNN 예측 결과 이미지 확인

정답과 예측값을 이미지 위에 함께 표시한다.  
틀린 예측은 빨간색으로 표시한다.

In [ ]:
show_images_labels(test_loader_cnn, classes, net=cnn_net, device=device, n_show=20)

## 27. 최종 평가 리포트

classification report와 confusion matrix로 class별 성능을 확인한다.

In [ ]:
def collect_predictions(net, loader, device):
    net.eval()

    y_true = []
    y_pred = []

    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            outputs = net(inputs)
            predicted = torch.max(outputs, 1)[1].cpu().numpy()

            y_pred.extend(predicted)
            y_true.extend(labels.numpy())

    return np.array(y_true), np.array(y_pred)

y_true, y_pred = collect_predictions(cnn_net, test_loader_cnn, device)

print(classification_report(y_true, y_pred, target_names=classes, zero_division=0))

cm = confusion_matrix(y_true, y_pred)
print(cm)

In [ ]:
plt.imshow(cm)
plt.title("CNN CIFAR-10 Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

for (i, j), value in np.ndenumerate(cm):
    if value > 0:
        plt.text(j, i, str(value), ha="center", va="center", fontsize=7)

plt.xticks(range(10), classes, rotation=45, ha="right")
plt.yticks(range(10), classes)
plt.colorbar()
plt.tight_layout()
plt.show()

## 28. Tensor 변환 연습: view와 permute

강의 마지막에는 Tensor shape을 바꾸는 연습이 나온다.

### 함수 사용법

```python
tensor.view(-1)
```

- Tensor를 1차원으로 펼친다.

```python
tensor.permute(1, 2, 0)
```

- 차원 순서를 바꾼다.
- PyTorch의 CHW를 이미지 출력용 HWC로 바꿀 때 자주 쓴다.

In [ ]:
sample_image = np.array([
    [[0, 0, 1],
     [0, 1, 0],
     [1, 0, 0]]
])

sample_tensor = torch.tensor(sample_image).float()

print("원본 shape:", sample_tensor.shape)
print("view(-1):", sample_tensor.view(-1).shape)
print("permute(1,2,0):", sample_tensor.permute(1, 2, 0).shape)
print("ndim:", sample_tensor.ndim)

## 29. 주요 함수 / 변수 / 약어 정리

| 이름 | 뜻 | 어떻게 쓰는지 |
|---|---|---|
| `CNN` | Convolutional Neural Network | 이미지에 특화된 신경망 |
| `FCN` | Fully Connected Network | 완전 결합형 신경망 |
| `Conv2d` | 2D 합성곱 Layer | `nn.Conv2d(in_ch, out_ch, kernel)` |
| `kernel` | 작은 필터 행렬 | 이미지 위를 이동하며 특징 추출 |
| `filter` | kernel과 비슷한 의미 | feature map 하나를 만드는 관점 |
| `feature map` | 필터 적용 결과 | 특징이 강조된 출력 |
| `channel` | 색상/특징 축 | RGB는 3채널 |
| `NCHW` | PyTorch CNN 입력 순서 | batch, channel, height, width |
| `MaxPool2d` | 최대값 풀링 | 공간 크기 축소 |
| `Flatten` | 1차원 펼치기 | classifier 입력으로 변환 |
| `features` | 특징 추출기 | Conv/ReLU/Pool 묶음 |
| `classifier` | 분류기 | Flatten/Linear 묶음 |
| `Sequential` | Layer 순서 묶음 | `nn.Sequential(...)` |
| `DataLoader` | batch 공급 도구 | `DataLoader(dataset, batch_size=...)` |
| `shuffle` | 데이터 순서 섞기 | train은 True |
| `view` | shape 변경 | `x.view(-1)` |
| `permute` | 차원 순서 변경 | CHW → HWC 변환 |

## 30. 시험용 요약

```text
CNN = 이미지의 공간 구조를 유지하면서 필터로 특징을 추출하는 신경망
```

꼭 기억할 것:

- DNN/FCN은 이미지를 1차원 벡터로 펼친다.
- 이미지를 펼치면 픽셀 간 공간 정보가 약해진다.
- CNN은 이미지 구조를 유지한 채 작은 필터로 특징을 뽑는다.
- CNN은 크게 특징 추출기와 분류기로 나뉜다.
- 특징 추출기는 `Conv2d → ReLU → Pooling` 구조다.
- 분류기는 `Flatten → Linear` 구조다.
- `Conv2d(in_channels, out_channels, kernel_size)` 형태로 쓴다.
- `out_channels`는 만들어질 feature map 개수다.
- `kernel_size=3`은 3×3 필터를 의미한다.
- PyTorch CNN 입력은 `[N, C, H, W]`다.
- RGB 이미지는 channel이 3개다.
- MNIST는 흑백이라 channel이 1개다.
- CIFAR-10은 RGB 32×32 이미지라 `[3, 32, 32]`이다.
- Padding 없이 3×3 Conv를 통과하면 H와 W가 2씩 줄어든다.
- `MaxPool2d(2,2)`는 보통 H와 W를 절반으로 줄인다.
- `Flatten`은 batch 차원을 제외하고 나머지를 1차원으로 펼친다.
- `[32, 14, 14]`를 Flatten하면 6272개 feature다.
- CNN은 이미지 분류에서 FCN보다 공간 정보를 잘 보존한다.
- `permute(1,2,0)`은 CHW 이미지를 HWC로 바꿔 시각화할 때 자주 쓴다.